# 01 · Serialize → dict / JSON

The heart of pyterraplot is `serialize(da)`, exposed as `da.tp.to_dict()` and `da.tp.to_json(path)`.
It converts a **2D** DataArray into terraplot's field contract:

```json
{ "lons": [...], "lats": [...], "field": [[...], ...], "name": ..., "units": ..., "long_name": ... }
```

`field` is row-major `field[j][i]` with `j` = latitude index, `i` = longitude index.

In [ ]:
import numpy as np
import xarray as xr
import pyterraplot  # registers the .tp accessor on DataArray and Dataset

def make_field(nlat=73, nlon=144, phase=0.0, name="t2m",
               long_name="2m temperature anomaly", units="K", holes=True):
    """A smooth, globe-shaped synthetic field on a regular lat/lon grid."""
    lats = np.linspace(90, -90, nlat)
    lons = np.linspace(-180, 180, nlon)
    LON, LAT = np.meshgrid(lons, lats)
    data = (
        8 * np.cos(np.radians(LAT)) * np.sin(np.radians(2 * LON) + phase)
        + 5 * np.sin(np.radians(3 * LON)) * np.cos(np.radians(2 * LAT))
        + 3 * np.cos(np.radians(5 * LON)) * np.sin(np.radians(LAT))
    ).astype(np.float32)
    if holes:
        rng = np.random.default_rng(0)
        data[rng.random((nlat, nlon)) < 0.02] = np.nan  # NaN "missing" cells
    return xr.DataArray(
        data, dims=["lat", "lon"], coords={"lat": lats, "lon": lons},
        name=name, attrs={"units": units, "long_name": long_name},
    )

da = make_field()
da

## `to_dict()` — the raw payload

Use this to build your own FastAPI/Flask route, or to inspect what gets sent to the browser.

In [ ]:
payload = da.tp.to_dict()
print("keys      :", list(payload))
print("lons      :", len(payload["lons"]), "values",
      f"[{payload['lons'][0]:.1f} … {payload['lons'][-1]:.1f}]")
print("lats      :", len(payload["lats"]), "values",
      f"[{payload['lats'][0]:.1f} … {payload['lats'][-1]:.1f}]")
print("field     :", len(payload["field"]), "×", len(payload["field"][0]))
print("name/units:", payload["name"], "/", payload["units"])
print("long_name :", payload["long_name"])

## NaN → `null`

NaN cells become JSON `null`, which terraplot renders transparent. Our synthetic field has ~2% holes punched in it.

In [ ]:
none_cells = sum(1 for row in payload["field"] for v in row if v is None)
print(f"{none_cells} null cells out of {da.size} "
      f"({100*none_cells/da.size:.1f}%)")

## `to_json()` — write a file

The browser fetches this with a plain `fetch()`.

In [ ]:
from pathlib import Path
p = da.tp.to_json("field.json")
print("wrote", p, f"({Path(p).stat().st_size/1024:.1f} kB)")

import json
reloaded = json.loads(Path(p).read_text())
print("round-trips:", reloaded.keys())

## Longitude wrapping (0→360 → −180→180)

Many climate datasets store longitude on `0→360`. By default `serialize` re-wraps to `−180→180` and re-sorts so the globe seams correctly. Disable with `wrap_lon=False`.

In [ ]:
# Build a 0→360 version of the same field
lons360 = np.linspace(0, 357.5, da.sizes["lon"])
da360 = da.assign_coords(lon=lons360)

wrapped   = da360.tp.to_dict()                 # default wrap_lon=True
unwrapped = da360.tp.to_dict(wrap_lon=False)

print("input lon range  :", float(lons360.min()), "→", float(lons360.max()))
print("wrapped  lons[0], lons[-1]:", wrapped["lons"][0], wrapped["lons"][-1])
print("unwrapped lons[0], lons[-1]:", unwrapped["lons"][0], unwrapped["lons"][-1])

## Dimension auto-detection & overrides

Lat/lon dims are auto-detected from common names: `lat, latitude, y, rlat, nav_lat, nlat` / `lon, longitude, x, rlon, nav_lon, nlon`. Pass `lon_dim=` / `lat_dim=` to override for non-standard names.

In [ ]:
# Rename to non-standard dims, then point pyterraplot at them explicitly
weird = da.rename({"lat": "south_north", "lon": "west_east"})
try:
    weird.tp.to_dict()                      # auto-detect fails
except ValueError as e:
    print("auto-detect error:", str(e).splitlines()[0])

ok = weird.tp.to_dict(lat_dim="south_north", lon_dim="west_east")
print("explicit dims worked — field:", len(ok["field"]), "×", len(ok["field"][0]))

### CF-convention detection (optional)

With `pip install 'pyterraplot[cf]'`, axes are also detected from CF metadata (`axis`/`standard_name` attrs) even when dim names are exotic. Without cf_xarray installed this just no-ops, so the cell below degrades gracefully.

In [ ]:
try:
    import cf_xarray  # noqa: F401
    cf_da = weird.copy()
    cf_da["south_north"].attrs["axis"] = "Y"
    cf_da["west_east"].attrs["axis"] = "X"
    out = cf_da.tp.to_dict()  # detected via CF attrs, no explicit dims needed
    print("cf_xarray detected axes — field:", len(out["field"]), "×", len(out["field"][0]))
except ImportError:
    print("cf_xarray not installed — install pyterraplot[cf] to try CF auto-detection")

## Must be 2D

Reduce extra dims (time, ensemble member, level) first — see notebook 03 for animating over them.

In [ ]:
cube = da.expand_dims(time=3)
try:
    cube.tp.to_dict()
except ValueError as e:
    print("error:", str(e).splitlines()[0])
print("fix it:", cube.isel(time=0).tp.to_dict().keys())